# Robust Polarimetric ISAR Classification via Complex Feature Fusion
Research notebook for `Pol-ISARSpaceTarget-1.0`.
Primary task: six-class satellite recognition from full-polarimetric complex ISAR.
Auxiliary task: rotated component detection from the provided labels.
This notebook is designed to run in Google Colab or a VS Code notebook connected
to a Colab GPU runtime.



In [ ]:
import json
import math
import os
import random
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

try:
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )
except Exception:
    accuracy_score = None
    confusion_matrix = None
    f1_score = None
    precision_score = None
    recall_score = None
    roc_auc_score = None


print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        os.system("nvidia-smi")
    except Exception as exc:
        print("nvidia-smi unavailable:", exc)




## Configuration
Set `DATA_ROOT` to the extracted dataset folder that contains:
- `ImageSets/images/train`
- `ImageSets/labelTxt/train`
- `Annotations`
For Colab Drive, this is usually something like:
`/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0`



In [ ]:
@dataclass
class Config:
    data_root: str = "/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0"
    output_dir: str = "/content/outputs/polisar_complex_multitask"
    image_size: int = 128
    batch_size: int = 8
    num_workers: int = 0
    epochs: int = 40
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 1.0
    seed: int = 42
    amp: bool = True
    detection_weight: float = 0.30
    objectness_weight: float = 1.0
    det_class_weight: float = 1.0
    box_weight: float = 4.0
    angle_weight: float = 1.0
    early_stop_patience: int = 12
    resume_checkpoint: str = ""
    representation: str = "complex_fusion"
    log_every_batches: int = 10

    @property
    def stride(self) -> int:
        return 16

    @property
    def grid_size(self) -> int:
        return self.image_size // self.stride


cfg = Config()

# If running locally from the extracted workspace, this fallback may work.
local_candidate = Path.cwd() / "Pol_ISARSpaceTarget1.0"
if not Path(cfg.data_root).exists() and local_candidate.exists():
    cfg.data_root = str(local_candidate)

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))




## Dataset and Preprocessing
Each training sample groups four `.dat` files:
`HH`, `HV`, `VH`, `VV`
The raw `.dat` format is interleaved float32 real/imaginary values for a
`256 x 256` complex image.



In [ ]:
SATELLITE_CLASSES = ["Capella", "HISEA1", "QPSSAR", "Radarsat2", "StarLink", "Worldview4"]
COMPONENT_CLASSES = [
    "cuboid_body",
    "optical_lens",
    "panel_antenna",
    "parabolic_antenna",
    "plate_body",
    "plate_fitting",
    "polygon_body",
    "rod_fitting",
    "solar_array",
]
POLS = ["HH", "HV", "VH", "VV"]
SAT_TO_ID = {name: idx for idx, name in enumerate(SATELLITE_CLASSES)}
COMP_TO_ID = {name: idx for idx, name in enumerate(COMPONENT_CLASSES)}
DRIVE_DUPLICATE_RE = re.compile(r" \(\d+\)(?=\.[^.]+$)")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def infer_satellite_id(base_name: str) -> int:
    for sat, idx in SAT_TO_ID.items():
        if base_name.startswith(sat + "_"):
            return idx
    raise ValueError(f"Cannot infer satellite class from {base_name}")


def is_drive_duplicate_name(name: str) -> bool:
    return DRIVE_DUPLICATE_RE.search(name) is not None


def dataset_upload_report(data_root: str) -> None:
    root = Path(data_root)
    checks = [
        ("train dat", root / "ImageSets" / "images" / "train", "*.dat", 5256),
        ("valid dat", root / "ImageSets" / "images" / "valid", "*.dat", 648),
        ("test dat", root / "ImageSets" / "images" / "test", "*.dat", 648),
        ("train labels", root / "ImageSets" / "labelTxt" / "train", "*.txt", 1314),
        ("valid labels", root / "ImageSets" / "labelTxt" / "valid", "*.txt", 162),
        ("test labels", root / "ImageSets" / "labelTxt" / "test", "*.txt", 162),
    ]
    print("Dataset root:", root)
    for name, path, pattern, expected in checks:
        files = list(path.glob(pattern)) if path.exists() else []
        duplicate_files = [p for p in files if is_drive_duplicate_name(p.name)]
        clean_files = [p for p in files if not is_drive_duplicate_name(p.name)]
        status = "OK" if len(clean_files) == expected else "CHECK"
        print(
            f"{name:13s} total={len(files):5d} clean={len(clean_files):5d} "
            f"duplicates={len(duplicate_files):3d} expected={expected:5d} {status}"
        )
        if duplicate_files[:3]:
            print("  duplicate examples:", [p.name for p in duplicate_files[:3]])


def read_complex_dat(path: Path) -> np.ndarray:
    data = np.fromfile(path, dtype=np.float32)
    if data.size != 256 * 256 * 2:
        raise ValueError(f"Unexpected file size for {path}: {data.size} float32 values")
    complex_image = data[0::2] + 1j * data[1::2]
    return complex_image.reshape(256, 256).astype(np.complex64)


def polygon_to_box(points: np.ndarray) -> Tuple[float, float, float, float, float]:
    center = points.mean(axis=0)
    edge01 = points[1] - points[0]
    edge12 = points[2] - points[1]
    width = float(np.linalg.norm(edge01))
    height = float(np.linalg.norm(edge12))
    angle = float(math.atan2(edge01[1], edge01[0]))
    return float(center[0]), float(center[1]), width, height, angle


def parse_label_file(path: Path) -> List[Dict[str, float]]:
    objects: List[Dict[str, float]] = []
    if not path.exists():
        return objects
    for raw in path.read_text().splitlines():
        parts = raw.strip().split()
        if len(parts) < 10:
            continue
        coords = np.array([float(x) for x in parts[:8]], dtype=np.float32).reshape(4, 2)
        class_name = parts[8]
        if class_name not in COMP_TO_ID:
            continue
        cx, cy, w, h, angle = polygon_to_box(coords)
        objects.append(
            {
                "cx": cx,
                "cy": cy,
                "w": w,
                "h": h,
                "angle": angle,
                "class_id": float(COMP_TO_ID[class_name]),
            }
        )
    return objects


def resize_complex_tensor(x: torch.Tensor, image_size: int) -> torch.Tensor:
    if x.shape[-1] == image_size and x.shape[-2] == image_size:
        return x
    real = F.interpolate(x.real.unsqueeze(0), size=(image_size, image_size), mode="bilinear", align_corners=False)
    imag = F.interpolate(x.imag.unsqueeze(0), size=(image_size, image_size), mode="bilinear", align_corners=False)
    return torch.complex(real.squeeze(0), imag.squeeze(0))


def normalize_complex_per_channel(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Scale each complex polarization so real/imag values stay near [-1, 1]."""
    max_amp = torch.abs(x).amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    return x / max_amp.to(x.dtype)


def magnitude_phase_tensor(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Return [mag, phase] features for HH/HV/VH/VV normalized to [-1, 1]."""
    mag = torch.abs(x)
    mag_max = mag.amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    mag_norm = 2.0 * (mag / mag_max) - 1.0
    phase_norm = torch.angle(x) / math.pi
    return torch.cat([mag_norm, phase_norm], dim=1).float()


def build_detection_target(
    objects: Sequence[Dict[str, float]], image_size: int, grid_size: int, num_classes: int
) -> torch.Tensor:
    target = torch.zeros(grid_size, grid_size, 1 + num_classes + 6, dtype=torch.float32)
    scale = image_size / 256.0
    cell = image_size / grid_size

    for obj in objects:
        cx = float(obj["cx"]) * scale
        cy = float(obj["cy"]) * scale
        w = max(float(obj["w"]) * scale, 1.0)
        h = max(float(obj["h"]) * scale, 1.0)
        angle = float(obj["angle"])
        gx = min(max(int(cx / cell), 0), grid_size - 1)
        gy = min(max(int(cy / cell), 0), grid_size - 1)
        existing_area = target[gy, gx, 1 + num_classes + 2] * target[gy, gx, 1 + num_classes + 3]
        area = (w / image_size) * (h / image_size)
        if target[gy, gx, 0] > 0 and existing_area >= area:
            continue

        target[gy, gx, 0] = 1.0
        class_id = int(obj["class_id"])
        target[gy, gx, 1 + class_id] = 1.0
        offset_x = (cx / cell) - gx
        offset_y = (cy / cell) - gy
        box_start = 1 + num_classes
        target[gy, gx, box_start : box_start + 6] = torch.tensor(
            [
                offset_x,
                offset_y,
                w / image_size,
                h / image_size,
                math.sin(angle),
                math.cos(angle),
            ],
            dtype=torch.float32,
        )

    return target


class PolISARDataset(Dataset):
    def __init__(self, data_root: str, split: str, image_size: int, grid_size: int) -> None:
        self.data_root = Path(data_root)
        self.split = split
        self.image_size = image_size
        self.grid_size = grid_size
        self.image_dir = self.data_root / "ImageSets" / "images" / split
        self.label_dir = self.data_root / "ImageSets" / "labelTxt" / split
        if not self.image_dir.exists():
            raise FileNotFoundError(f"Missing image split directory: {self.image_dir}")
        if not self.label_dir.exists():
            raise FileNotFoundError(f"Missing label split directory: {self.label_dir}")

        duplicate_dat = [p.name for p in self.image_dir.glob("*.dat") if is_drive_duplicate_name(p.name)]
        duplicate_txt = [p.name for p in self.label_dir.glob("*.txt") if is_drive_duplicate_name(p.name)]
        if duplicate_dat or duplicate_txt:
            print(
                f"[{split}] Ignoring Drive duplicate files: "
                f"{len(duplicate_dat)} dat, {len(duplicate_txt)} labels"
            )

        bases = sorted(
            {
                p.name.rsplit("_", 1)[0]
                for p in self.image_dir.glob("*_HH.dat")
                if not is_drive_duplicate_name(p.name)
            }
        )
        self.samples = [base for base in bases if all((self.image_dir / f"{base}_{pol}.dat").exists() for pol in POLS)]
        if not self.samples:
            raise RuntimeError(f"No complete HH/HV/VH/VV samples found in {self.image_dir}")
        expected = {"train": 1314, "valid": 162, "test": 162}.get(split)
        if expected is not None and len(self.samples) != expected:
            print(f"[{split}] Warning: usable complete samples={len(self.samples)}, expected={expected}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        base = self.samples[index]
        channels = []
        for pol in POLS:
            arr = read_complex_dat(self.image_dir / f"{base}_{pol}.dat")
            channels.append(torch.from_numpy(arr))
        x = torch.stack(channels, dim=0)
        x = resize_complex_tensor(x, self.image_size)
        x = normalize_complex_per_channel(x)

        label_path = self.label_dir / f"{base}.txt"
        objects = parse_label_file(label_path)
        det_target = build_detection_target(objects, self.image_size, self.grid_size, len(COMPONENT_CLASSES))
        return {
            "x": x,
            "class_id": torch.tensor(infer_satellite_id(base), dtype=torch.long),
            "det_target": det_target,
            "base": base,
        }


def collate_batch(batch: Sequence[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    return {
        "x": torch.stack([item["x"] for item in batch]),
        "class_id": torch.stack([item["class_id"] for item in batch]),
        "det_target": torch.stack([item["det_target"] for item in batch]),
        "base": [str(item["base"]) for item in batch],
    }


seed_everything(cfg.seed)
dataset_upload_report(cfg.data_root)
train_ds = PolISARDataset(cfg.data_root, "train", cfg.image_size, cfg.grid_size)
valid_ds = PolISARDataset(cfg.data_root, "valid", cfg.image_size, cfg.grid_size)
test_ds = PolISARDataset(cfg.data_root, "test", cfg.image_size, cfg.grid_size)
print("Dataset sizes:", len(train_ds), len(valid_ds), len(test_ds))
sample = train_ds[0]
print("Sample:", sample["base"], sample["x"].shape, sample["x"].dtype, sample["det_target"].shape)
sample_mp = magnitude_phase_tensor(sample["x"].unsqueeze(0))
print(
    "Magnitude-phase feature range:",
    float(sample_mp.min()),
    float(sample_mp.max()),
    "shape:",
    tuple(sample_mp.shape),
)




## Complex CNN Layers
Complex convolution is implemented from the mathematical product:
`(a + jb)(c + jd) = (ac - bd) + j(ad + bc)`
BatchNorm is applied separately to real and imaginary components. This is a
computationally efficient approximation; a full covariance complex BN is a
possible improvement for a later ablation.



In [ ]:
class ComplexConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int = 1,
        padding: int = 0,
        bias: bool = False,
    ) -> None:
        super().__init__()
        self.real = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)
        self.imag = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        real = self.real(x.real) - self.imag(x.imag)
        imag = self.real(x.imag) + self.imag(x.real)
        return torch.complex(real, imag)


class ComplexBatchNorm2d(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.real_bn = nn.BatchNorm2d(channels)
        self.imag_bn = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.complex(self.real_bn(x.real), self.imag_bn(x.imag))


class ComplexModReLU(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        magnitude = torch.abs(x)
        activated = F.relu(magnitude + self.bias)
        phase = x / (magnitude + self.eps)
        return activated.to(x.dtype) * phase


class ComplexAvgPool2d(nn.Module):
    def __init__(self, kernel_size: int, stride: int) -> None:
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size, stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.complex(self.pool(x.real), self.pool(x.imag))


class ComplexResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = ComplexConv2d(in_channels, out_channels, 3, stride=stride, padding=1)
        self.bn1 = ComplexBatchNorm2d(out_channels)
        self.act1 = ComplexModReLU(out_channels)
        self.conv2 = ComplexConv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = ComplexBatchNorm2d(out_channels)
        self.act2 = ComplexModReLU(out_channels)
        self.proj: Optional[nn.Module]
        if stride != 1 or in_channels != out_channels:
            self.proj = nn.Sequential(
                ComplexConv2d(in_channels, out_channels, 1, stride=stride),
                ComplexBatchNorm2d(out_channels),
            )
        else:
            self.proj = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if self.proj is None else self.proj(x)
        out = self.act1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act2(out + identity)


class ComplexSEAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8) -> None:
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mag = torch.abs(x).mean(dim=(-2, -1))
        weights = torch.sigmoid(self.fc2(F.relu(self.fc1(mag)))).unsqueeze(-1).unsqueeze(-1)
        return x * weights.to(x.dtype)


class ComplexSpatialAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mag = torch.abs(x)
        avg_map = mag.mean(dim=1, keepdim=True)
        max_map = mag.max(dim=1, keepdim=True).values
        attn = torch.sigmoid(self.conv(torch.cat([avg_map, max_map], dim=1)))
        return x * attn.to(x.dtype)


class ComplexFusionBlock(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.channel_attn = ComplexSEAttention(channels)
        self.spatial_attn = ComplexSpatialAttention()
        self.refine = ComplexResidualBlock(channels, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.channel_attn(x)
        out = self.spatial_attn(out)
        return self.refine(out + x)


class MagnitudePhaseBranch(nn.Module):
    """Real-valued branch that explicitly taps normalized magnitude and phase."""

    def __init__(self, out_dim: int = 128) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 96, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96, out_dim),
            nn.Tanh(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mp = magnitude_phase_tensor(x)
        return self.proj(self.encoder(mp))




## Multi-Task Complex CNN
The shared backbone learns phase-preserving polarimetric features.
Guide-required classification path:
1. Use a Complex CNN over the full HH/HV/VH/VV complex tensor.
2. Tap explicit magnitude and phase features from the same tensor.
3. Normalize magnitude and phase features to `[-1, 1]`.
4. Project complex and magnitude-phase streams to bounded feature vectors.
5. Concatenate both vectors and feed them to the classification head.
6. Compute classification metrics on train/validation/test splits.
The classification head is the primary branch. The detection head is auxiliary
and encourages component-aware features.



In [ ]:
class ComplexPolISARNet(nn.Module):
    def __init__(self, num_sat_classes: int, num_component_classes: int) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            ComplexConv2d(4, 32, kernel_size=5, stride=2, padding=2),
            ComplexBatchNorm2d(32),
            ComplexModReLU(32),
        )
        self.stage1 = nn.Sequential(
            ComplexResidualBlock(32, 48, stride=2),
            ComplexResidualBlock(48, 48),
        )
        self.stage2 = nn.Sequential(
            ComplexResidualBlock(48, 96, stride=2),
            ComplexResidualBlock(96, 96),
        )
        self.stage3 = nn.Sequential(
            ComplexResidualBlock(96, 160, stride=2),
            ComplexResidualBlock(160, 160),
        )
        self.fusion = ComplexFusionBlock(160)
        self.complex_vector = nn.Sequential(
            nn.Linear(320, 192),
            nn.GELU(),
            nn.Linear(192, 128),
            nn.Tanh(),
        )
        self.mag_phase_branch = MagnitudePhaseBranch(out_dim=128)
        self.classifier = nn.Sequential(
            nn.Linear(256, 192),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(192, num_sat_classes),
        )
        det_channels = 1 + num_component_classes + 6
        self.det_head = nn.Sequential(
            nn.Conv2d(320, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.GELU(),
            nn.Conv2d(192, det_channels, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        mag_phase_vector = self.mag_phase_branch(x)
        z = self.stem(x)
        z = self.stage1(z)
        z = self.stage2(z)
        z = self.stage3(z)
        z = self.fusion(z)
        real_features = torch.cat([z.real, z.imag], dim=1)
        complex_pooled = real_features.mean(dim=(-2, -1))
        complex_vector = self.complex_vector(complex_pooled)
        fused_vector = torch.cat([complex_vector, mag_phase_vector], dim=1)
        class_logits = self.classifier(fused_vector)
        det_logits = self.det_head(real_features).permute(0, 2, 3, 1).contiguous()
        return {
            "class_logits": class_logits,
            "det_logits": det_logits,
            "complex_vector": complex_vector,
            "mag_phase_vector": mag_phase_vector,
            "fused_vector": fused_vector,
        }


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
if device.type == "cuda":
    print("Using GPU:", torch.cuda.get_device_name(0))
model = ComplexPolISARNet(len(SATELLITE_CLASSES), len(COMPONENT_CLASSES)).to(device)
print(model)
print("Trainable parameters:", f"{count_parameters(model):,}")

with torch.no_grad():
    smoke_x = sample["x"].unsqueeze(0).to(device)
    smoke_out = model(smoke_x)
    print("Smoke class logits:", tuple(smoke_out["class_logits"].shape))
    print("Smoke det logits:", tuple(smoke_out["det_logits"].shape))
    print("Complex vector range:", float(smoke_out["complex_vector"].min()), float(smoke_out["complex_vector"].max()))
    print("Mag-phase vector range:", float(smoke_out["mag_phase_vector"].min()), float(smoke_out["mag_phase_vector"].max()))
    print("Fused vector:", tuple(smoke_out["fused_vector"].shape))




## Losses and Metrics



In [ ]:
class MultiTaskLoss(nn.Module):
    def __init__(self, config: Config) -> None:
        super().__init__()
        self.cfg = config
        self.cls_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
        self.obj_loss = nn.BCEWithLogitsLoss()
        self.det_cls_loss = nn.CrossEntropyLoss()
        self.box_loss = nn.SmoothL1Loss()

    def forward(
        self, outputs: Dict[str, torch.Tensor], class_target: torch.Tensor, det_target: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        class_loss = self.cls_loss(outputs["class_logits"], class_target)
        det_logits = outputs["det_logits"]
        obj_target = det_target[..., 0]
        obj_loss = self.obj_loss(det_logits[..., 0], obj_target)

        positive = obj_target > 0.5
        if positive.any():
            det_class_logits = det_logits[..., 1 : 1 + len(COMPONENT_CLASSES)][positive]
            det_class_target = det_target[..., 1 : 1 + len(COMPONENT_CLASSES)][positive].argmax(dim=-1)
            det_class_loss = self.det_cls_loss(det_class_logits, det_class_target)

            box_start = 1 + len(COMPONENT_CLASSES)
            pred_box = det_logits[..., box_start : box_start + 6][positive]
            target_box = det_target[..., box_start : box_start + 6][positive]
            pred_box = pred_box.clone()
            pred_box[:, 0:2] = torch.sigmoid(pred_box[:, 0:2])
            pred_box[:, 2:4] = torch.sigmoid(pred_box[:, 2:4])
            pred_angle = F.normalize(pred_box[:, 4:6], dim=-1)
            target_angle = F.normalize(target_box[:, 4:6], dim=-1)
            box_loss = self.box_loss(pred_box[:, :4], target_box[:, :4])
            angle_loss = self.box_loss(pred_angle, target_angle)
        else:
            det_class_loss = det_logits.sum() * 0.0
            box_loss = det_logits.sum() * 0.0
            angle_loss = det_logits.sum() * 0.0

        detection_loss = (
            self.cfg.objectness_weight * obj_loss
            + self.cfg.det_class_weight * det_class_loss
            + self.cfg.box_weight * box_loss
            + self.cfg.angle_weight * angle_loss
        )
        total = class_loss + self.cfg.detection_weight * detection_loss
        return {
            "loss": total,
            "class_loss": class_loss.detach(),
            "detection_loss": detection_loss.detach(),
            "objectness_loss": obj_loss.detach(),
            "det_class_loss": det_class_loss.detach(),
            "box_loss": box_loss.detach(),
            "angle_loss": angle_loss.detach(),
        }


@torch.no_grad()
def classification_metrics(y_true: Sequence[int], logits: torch.Tensor) -> Dict[str, float]:
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    y_pred = probs.argmax(axis=1)
    y_true_np = np.array(y_true)
    metrics = {
        "accuracy": float((y_pred == y_true_np).mean()),
    }
    if accuracy_score is not None:
        metrics.update(
            {
                "precision_macro": float(precision_score(y_true_np, y_pred, average="macro", zero_division=0)),
                "recall_macro": float(recall_score(y_true_np, y_pred, average="macro", zero_division=0)),
                "f1_macro": float(f1_score(y_true_np, y_pred, average="macro", zero_division=0)),
            }
        )
        try:
            metrics["auc_ovr"] = float(roc_auc_score(y_true_np, probs, multi_class="ovr"))
        except Exception:
            metrics["auc_ovr"] = float("nan")
    return metrics




## Training and Evaluation



In [ ]:
def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_batch,
        persistent_workers=False,
    )


def move_batch(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {
        "x": batch["x"].to(device, non_blocking=True),
        "class_id": batch["class_id"].to(device, non_blocking=True),
        "det_target": batch["det_target"].to(device, non_blocking=True),
        "base": batch["base"],
    }


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    epoch: int,
    best_metric: float,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_metric": best_metric,
            "config": asdict(cfg),
            "satellite_classes": SATELLITE_CLASSES,
            "component_classes": COMPONENT_CLASSES,
        },
        path,
    )


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
) -> Tuple[int, float]:
    checkpoint = torch.load(path, map_location="cpu")
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None and "scheduler" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler"])
    return int(checkpoint.get("epoch", 0)) + 1, float(checkpoint.get("best_metric", 0.0))


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: MultiTaskLoss,
    optimizer: Optional[torch.optim.Optimizer],
    scaler: Optional[torch.amp.GradScaler],
    device: torch.device,
    phase: str,
) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    totals: Dict[str, float] = {}
    n_batches = 0
    all_targets: List[int] = []
    all_logits: List[torch.Tensor] = []
    correct = 0
    seen = 0

    iterator: Iterable[Dict[str, torch.Tensor]] = loader
    total_batches = len(loader)
    print(f"{phase}: starting {total_batches} batches", flush=True)

    for batch_idx, batch in enumerate(iterator, start=1):
        batch = move_batch(batch, device)
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=cfg.amp and torch.cuda.is_available()):
                outputs = model(batch["x"])
                losses = criterion(outputs, batch["class_id"], batch["det_target"])

            if is_train:
                assert scaler is not None
                scaler.scale(losses["loss"]).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()

        for key, value in losses.items():
            totals[key] = totals.get(key, 0.0) + float(value.detach().cpu())
        n_batches += 1
        preds = outputs["class_logits"].detach().argmax(dim=1)
        correct += int((preds == batch["class_id"]).sum().item())
        seen += int(batch["class_id"].numel())
        all_targets.extend(batch["class_id"].detach().cpu().tolist())
        all_logits.append(outputs["class_logits"].detach().cpu())

        should_log = (
            batch_idx == 1
            or batch_idx == total_batches
            or batch_idx % max(cfg.log_every_batches, 1) == 0
        )
        if should_log:
            print(
                f"{phase} [{batch_idx:04d}/{total_batches:04d}] "
                f"loss={totals['loss'] / n_batches:.4f} "
                f"cls={totals['class_loss'] / n_batches:.4f} "
                f"det={totals['detection_loss'] / n_batches:.4f} "
                f"acc={correct / max(seen, 1):.3f}",
                flush=True,
            )

    avg = {key: value / max(n_batches, 1) for key, value in totals.items()}
    logits = torch.cat(all_logits, dim=0)
    avg.update(classification_metrics(all_targets, logits))
    return avg


train_loader = make_loader(train_ds, shuffle=True)
valid_loader = make_loader(valid_ds, shuffle=False)
test_loader = make_loader(test_ds, shuffle=False)

criterion = MultiTaskLoss(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and torch.cuda.is_available())

start_epoch = 0
best_valid_f1 = 0.0
if cfg.resume_checkpoint:
    start_epoch, best_valid_f1 = load_checkpoint(cfg.resume_checkpoint, model, optimizer, scheduler)
    print("Resumed from", cfg.resume_checkpoint, "start_epoch", start_epoch, "best_valid_f1", best_valid_f1)

history: List[Dict[str, float]] = []
best_path = Path(cfg.output_dir) / "best_complex_polisar_multitask.pt"
last_path = Path(cfg.output_dir) / "last_complex_polisar_multitask.pt"
epochs_without_improvement = 0
print("Checkpoints:")
print("  best:", best_path)
print("  last:", last_path)
print("Early stopping monitors validation macro-F1, fallback validation accuracy.")

for epoch in range(start_epoch, cfg.epochs):
    t0 = time.time()
    train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler, device, phase=f"epoch {epoch + 1} train")
    valid_metrics = run_epoch(model, valid_loader, criterion, None, None, device, phase=f"epoch {epoch + 1} valid")
    scheduler.step()

    row = {
        "epoch": float(epoch),
        "lr": float(scheduler.get_last_lr()[0]),
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"valid_{k}": v for k, v in valid_metrics.items()},
        "seconds": time.time() - t0,
    }
    history.append(row)
    print(json.dumps(row, indent=2))

    current = valid_metrics.get("f1_macro", valid_metrics["accuracy"])
    save_checkpoint(last_path, model, optimizer, scheduler, epoch, best_valid_f1)
    if current > best_valid_f1:
        best_valid_f1 = current
        epochs_without_improvement = 0
        save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_valid_f1)
        print(f"Saved best checkpoint: {best_path} with monitored score={best_valid_f1:.4f}")
    else:
        epochs_without_improvement += 1
        print(
            f"No validation improvement for {epochs_without_improvement}/"
            f"{cfg.early_stop_patience} epochs."
        )

    if epochs_without_improvement >= cfg.early_stop_patience:
        print("Early stopping triggered.")
        break

history_path = Path(cfg.output_dir) / "history.json"
history_path.write_text(json.dumps(history, indent=2))
print("Training history saved:", history_path)




## Final Test Evaluation



In [ ]:
if best_path.exists():
    load_checkpoint(str(best_path), model)
    print("Loaded best checkpoint for test evaluation.")

test_metrics = run_epoch(model, test_loader, criterion, None, None, device, phase="test")
print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

all_targets: List[int] = []
all_preds: List[int] = []
all_probs: List[np.ndarray] = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = move_batch(batch, device)
        outputs = model(batch["x"])
        probs = torch.softmax(outputs["class_logits"], dim=1).cpu().numpy()
        all_probs.extend(list(probs))
        all_preds.extend(probs.argmax(axis=1).tolist())
        all_targets.extend(batch["class_id"].cpu().tolist())

if confusion_matrix is not None:
    cm = confusion_matrix(all_targets, all_preds)
    print("Confusion matrix rows=true cols=pred:")
    print(cm)
    np.save(Path(cfg.output_dir) / "test_confusion_matrix.npy", cm)

with open(Path(cfg.output_dir) / "test_predictions.json", "w") as f:
    json.dump(
        {
            "targets": all_targets,
            "predictions": all_preds,
            "classes": SATELLITE_CLASSES,
            "metrics": test_metrics,
        },
        f,
        indent=2,
    )

print("Outputs written to:", cfg.output_dir)




## Inference Timing



In [ ]:
@torch.no_grad()
def benchmark_inference(model: nn.Module, image_size: int, repeats: int = 100) -> Dict[str, float]:
    model.eval()
    x = torch.randn(1, 4, image_size, image_size, device=device)
    x = torch.complex(x, torch.randn_like(x))
    for _ in range(10):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeats):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start
    return {"ms_per_sample": 1000.0 * elapsed / repeats}


print(benchmark_inference(model, cfg.image_size))
